In [ ]:
import os
import sys
import torch
import torch.optim as optim
import matplotlib.pyplot as plt

sys.path.append("..")

from src.data_utils import load_mnist_vae_data, get_device, seed_everything
from src.models import ConvVAE
from src.train import train_vae
from src.evaluate import evaluate_vae, reconstruct_images, sample_from_prior
from src.visualize import (
    plot_training_curves,
    show_reconstructions,
    show_generated_samples,
    plot_latent_scatter,
    plot_latent_grid,
)
from src.latent_analysis import encode_dataset

In [ ]:
seed_everything(42)
device = get_device()
print("device:", device)

os.makedirs("../results/figures", exist_ok=True)
os.makedirs("../results/logs", exist_ok=True)

In [ ]:
train_loader, val_loader, test_loader = load_mnist_vae_data(
    data_dir="../data",
    batch_size=512,
    val_ratio=0.1,
    num_workers=0,
    random_state=42,
)

print("train batches:", len(train_loader))
print("val batches:", len(val_loader))
print("test batches:", len(test_loader))

In [ ]:
x_batch, y_batch = next(iter(train_loader))
print("x_batch shape:", x_batch.shape)
print("y_batch shape:", y_batch.shape)

fig, axes = plt.subplots(1, 8, figsize=(14, 2))
for i in range(8):
    axes[i].imshow(x_batch[i].squeeze(), cmap="gray")
    axes[i].set_title(str(y_batch[i].item()))
    axes[i].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
import torch.optim as optim

latent_dim = 16
beta = 1.0
epochs = 20
lr = 1e-3
hidden_dim = 256
base_channels = 64
batch_size = 512
warmup_epochs = None
recon_loss_type = "bce"

model = ConvVAE(
    input_channels=1,
    latent_dim=latent_dim,
    hidden_dim=hidden_dim,
    base_channels=base_channels,
).to(device)

optimizer = optim.Adam(model.parameters(), lr=lr)

In [ ]:
save_path = f"../results/logs/h1_conv_vae_latent{latent_dim}.pt"

history = train_vae(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=device,
    epochs=epochs,
    beta=beta,
    recon_loss_type="bce",
    save_path=save_path,
    warmup_epochs=warmup_epochs,
)

In [ ]:
plot_training_curves(
    history,
    save_path=f"../results/figures/h1_training_curves_latent{latent_dim}.png"
)

In [ ]:
test_metrics = evaluate_vae(
    model,
    test_loader,
    device=device,
    beta=beta,
    recon_loss_type=recon_loss_type,
)

print("Test metrics:")
for k, v in test_metrics.items():
    print(f"{k}: {v:.4f}")

In [ ]:
x_test, y_test = next(iter(test_loader))
recon_x, mu, logvar = reconstruct_images(model, x_test[:8], device=device)

show_reconstructions(
    x_test[:8].cpu(),
    recon_x,
    n=8,
    save_path=f"../results/figures/h1_reconstructions_latent{latent_dim}.png"
)

In [ ]:
samples = sample_from_prior(model, n_samples=16, device=device)

show_generated_samples(
    samples,
    n=16,
    save_path=f"../results/figures/h1_generated_samples_latent{latent_dim}.png"
)

In [ ]:
encoded = encode_dataset(model, test_loader, device=device)
z_test = encoded["mu"]
y_test = encoded["y"]

print("z_test shape:", z_test.shape)

plot_latent_scatter(
    z_test,
    y_test,
    save_path=f"../results/figures/h1_latent_scatter_latent{latent_dim}.png"
)

In [ ]:
plot_latent_grid(
    model,
    device=device,
    grid_size=20,
    latent_range=(-3, 3),
    save_path=f"../results/figures/h1_latent_grid_latent{latent_dim}.png"
)

In [ ]:
result_summary = {
    "latent_dim": latent_dim,
    "beta": beta,
    "epochs": epochs,
    "lr": lr,
    "recon_loss_type": recon_loss_type,
    "test_loss": test_metrics["loss"],
    "test_recon_loss": test_metrics["recon_loss"],
    "test_kl_loss": test_metrics["kl_loss"],
}

result_summary

In [ ]:
best_config = {
    "latent_dim": 16,
    "beta": 1.0,
    "epochs": 20,
    "lr": 1e-3,
    "hidden_dim": 256,
    "base_channels": 64,
    "batch_size": 512,
    "warmup_epochs": None,
    "recon_loss_type": "bce",
}
best_config

In [ ]:
from pathlib import Path
import torch

save_path = Path("../results/logs/h1_conv_vae_best_latent16.pt")
save_path.parent.mkdir(parents=True, exist_ok=True)

torch.save(model.state_dict(), save_path)

print("saved to:", save_path.resolve())
print("exists:", save_path.exists())

In [ ]:
latent_dim = 2
beta = 1.0
epochs = 20
lr = 1e-3
save_path = "../results/logs/h1_conv_vae_best_latent2.pt"

model = ConvVAE(
    input_channels=1,
    latent_dim=latent_dim,
    hidden_dim=256,
    base_channels=64,
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=lr)

history = train_vae(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    device=device,
    epochs=epochs,
    beta=beta,
    recon_loss_type="bce",
    save_path=save_path,
    warmup_epochs=None,
)

In [ ]:
torch.save(model.state_dict(), "../results/logs/h1_conv_vae_latent2_last.pt")